In [ ]:
#ver 2

In [53]:
"""
Bus Fleet Dispatch and Frequency Optimisation
Small-sized Gurobi benchmark using frequency-based allocation.

Updated formulation:
- Decision is frequency / number of dispatches per service-period.
- Headway is derived from frequency: h = period_length / frequency.
- Demand is allocated across dispatches: demand_per_dispatch = demand / frequency.
- Overcrowding is calculated based on whether total demand exceeds total dispatch capacity.
- Fleet requirement is estimated from cycle time and selected frequency.
"""

import math
import time
from gurobipy import Model, GRB, quicksum


# ============================================================
# 1. Create small-sized Tampines example instance
# ============================================================

def create_small_instance():
    """
    Small illustrative instance.

    Replace the sample values with your processed LTA data:
    - D[i,t]: passenger demand from tap-in / passenger volume data
    - headway_options: observed or selected headway values from bus arrival data
    - cycle_time[i]: estimated round-trip cycle time for each service
    - q[i]: bus capacity, e.g. 70 for single-deck, 100 for double-deck
    """

    services = ["8", "18", "28", "38", "69"]

    periods = ["Morning_Peak", "Off_Peak", "Evening_Peak"]

    # Each time period is 2 hours = 120 minutes
    # Or maybe we can change it to 1 hour and then multiply by 2? since the LTA data is given per hour
    L = {
        "Morning_Peak": 120,
        "Off_Peak": 120,
        "Evening_Peak": 120,
    }

    # Available fleet during each time period
    # This is the number of physical buses available for the selected services. I got these numbers as a rough estimation
    available_fleet = {
        "Morning_Peak": 500,
        "Off_Peak": 300,
        "Evening_Peak": 500,
    }

    # Estimated passenger demand by service and period
    # Replace with LTA passenger volume data
    D = {
        ("8", "Morning_Peak"): 850,
        ("8", "Off_Peak"): 420,
        ("8", "Evening_Peak"): 780,

        ("18", "Morning_Peak"): 520,
        ("18", "Off_Peak"): 260,
        ("18", "Evening_Peak"): 500,

        ("28", "Morning_Peak"): 920,
        ("28", "Off_Peak"): 460,
        ("28", "Evening_Peak"): 900,

        ("38", "Morning_Peak"): 430,
        ("38", "Off_Peak"): 200,
        ("38", "Evening_Peak"): 410,

        ("69", "Morning_Peak"): 760,
        ("69", "Off_Peak"): 370,
        ("69", "Evening_Peak"): 730,
    }

    # Bus Capacity
    # 70 for single-deck and 100 for double-deck
    q = {
        "8": 100,
        "18": 70,
        "28": 100,
        "38": 70,
        "69": 100,
    }

    # Estimated round-trip cycle time in minutes
    # Used only to estimate physical buses required for a selected frequency
    cycle_time = {
        "8": 65,
        "18": 55,
        "28": 70,
        "38": 50,
        "69": 60,
    }

    # Candidate headway options
    # Comes from LTA bus arrival data (arrival time of bus 2 - arrival time of bus 1)
    # Example: If observed headways during AM peak are around 4, 5, 6, and 8 minutes, use [4, 5, 6, 8]
    headway_options = {}

    for i in services:
        for t in periods:
            if t in ["Morning_Peak", "Evening_Peak"]:
                headway_options[i, t] = [3, 4, 5, 7]
            else:
                headway_options[i, t] = [5, 6, 7.5, 8]

    return services, periods, L, available_fleet, D, q, cycle_time, headway_options


# ============================================================
# 2. Solve exact small-sized model using Gurobi
# ============================================================

def solve_frequency_based_gurobi(
    services,
    periods,
    L,
    available_fleet,
    D,
    q,
    cycle_time,
    headway_options,
    alpha=0.40,
    beta=0.50,
    gamma=0.10,
    time_limit=60,
    verbose=True,
):
    """
    Gurobi model.

    Binary decision variable:
        y[i,t,h] = 1 if service i in period t uses headway option h.

    Derived values:
        frequency = L[t] / h
        demand_per_dispatch = D[i,t] / frequency
        total_capacity = frequency * q[i]
        overcrowding = max(0, D[i,t] - total_capacity)
        waiting/headway penalty = (h - 5)^2 + 2
        buses_required = ceil(cycle_time[i] / h)

    Objective:
        Minimise weighted sum of:
        1. passenger-weighted waiting/headway penalty
        2. overcrowding penalty
        3. number of physical buses required
    """

    model = Model("frequency_based_bus_dispatch")

    if not verbose:
        model.Params.OutputFlag = 0

    model.Params.TimeLimit = time_limit

    # ------------------------------------------------------------
    # Precompute values for every service-period-headway option
    # ------------------------------------------------------------

    frequency = {}
    buses_required = {}
    demand_per_dispatch = {}
    total_capacity = {}
    overcrowding = {}
    overcrowding_penalty = {}
    headway_penalty = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:

                # Frequency = number of departures in the 2-hour period, which is also the no. of buses
                frequency[i, t, h] = L[t] / h

                # Demand allocated to each dispatch, since we only have the total tap-in value for each hour
                demand_per_dispatch[i, t, h] = D[i, t] / frequency[i, t, h]

                # Total capacity across all dispatches. this is essentially no. of buses * capacity of 1 bus
                total_capacity[i, t, h] = frequency[i, t, h] * q[i]

                # Overcrowding amount = when total demand exceeds total capacity
                overcrowding[i, t, h] = max(
                    0,
                    D[i, t] - total_capacity[i, t, h]
                )

                # Squared overcrowding penalty
                overcrowding_penalty[i, t, h] = overcrowding[i, t, h] ** 2

                # Quadratic headway penalty
                headway_penalty[i, t, h] = (h - 5) ** 2 + 2

                # Physical buses required to sustain that headway
                # It estimates fleet requirement from cycle time / selected headway
                buses_required[i, t, h] = math.ceil(cycle_time[i] / h)

    # ------------------------------------------------------------
    # Decision variables
    # ------------------------------------------------------------

    y = {}

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:
                y[i, t, h] = model.addVar(
                    vtype=GRB.BINARY,
                    name=f"choose_{i}_{t}_h{h}"
                )

    model.update()

    # ------------------------------------------------------------
    # Constraint 1:
    # Choose exactly one frequency/headway option for each service-period
    # ------------------------------------------------------------

    for i in services:
        for t in periods:
            model.addConstr(
                quicksum(y[i, t, h] for h in headway_options[i, t]) == 1,
                name=f"choose_one_option_{i}_{t}"
            )

    # ------------------------------------------------------------
    # Constraint 2:
    # Fleet availability during each time period
    # ------------------------------------------------------------

    for t in periods:
        model.addConstr(
            quicksum(
                buses_required[i, t, h] * y[i, t, h]
                for i in services
                for h in headway_options[i, t]
            ) <= available_fleet[t],
            name=f"fleet_limit_{t}"
        )

    # ------------------------------------------------------------
    # Normalisation
    # This avoids one objective component dominating only because of scale.
    # ------------------------------------------------------------

    max_waiting_component = sum(
        D[i, t] * max(headway_penalty[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_overcrowding_component = sum(
        max(overcrowding_penalty[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_bus_component = sum(
        max(buses_required[i, t, h] for h in headway_options[i, t])
        for i in services
        for t in periods
    )

    max_waiting_component = max(max_waiting_component, 1)
    max_overcrowding_component = max(max_overcrowding_component, 1)
    max_bus_component = max(max_bus_component, 1)

    # ------------------------------------------------------------
    # Objective components
    # ------------------------------------------------------------

    waiting_component = quicksum(
        D[i, t] * headway_penalty[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    overcrowding_component = quicksum(
        overcrowding_penalty[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    bus_component = quicksum(
        buses_required[i, t, h] * y[i, t, h]
        for i in services
        for t in periods
        for h in headway_options[i, t]
    )

    model.setObjective(
        alpha * waiting_component / max_waiting_component
        + beta * overcrowding_component / max_overcrowding_component
        + gamma * bus_component / max_bus_component,
        GRB.MINIMIZE
    )

    # ------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------

    start_time = time.time()
    model.optimize()
    runtime = time.time() - start_time

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print("No feasible solution found.")
        print("Gurobi status:", model.Status)
        return None

    # ------------------------------------------------------------
    # Extract solution
    # ------------------------------------------------------------

    solution = []

    raw_waiting = 0
    raw_overcrowding = 0
    raw_buses = 0

    for i in services:
        for t in periods:
            for h in headway_options[i, t]:
                if y[i, t, h].X > 0.5:

                    row = {
                        "service": i,
                        "period": t,
                        "headway_min": h,
                        "frequency_departures": frequency[i, t, h],
                        "buses_required": buses_required[i, t, h],
                        "demand": D[i, t],
                        "demand_per_dispatch": demand_per_dispatch[i, t, h],
                        "capacity_per_dispatch": q[i],
                        "total_capacity": total_capacity[i, t, h],
                        "overcrowding": overcrowding[i, t, h],
                        "headway_penalty": headway_penalty[i, t, h],
                        "overcrowding_penalty": overcrowding_penalty[i, t, h],
                    }

                    solution.append(row)

                    raw_waiting += D[i, t] * headway_penalty[i, t, h]
                    raw_overcrowding += overcrowding_penalty[i, t, h]
                    raw_buses += buses_required[i, t, h]

    result = {
        "status": model.Status,
        "objective": model.ObjVal,
        "runtime_seconds": runtime,
        "mip_gap": model.MIPGap,
        "raw_waiting_component": raw_waiting,
        "raw_overcrowding_component": raw_overcrowding,
        "raw_bus_component": raw_buses,
        "solution": solution,
    }

    return result


# ============================================================
# 3. Print solution
# ============================================================

def print_solution(result):
    if result is None:
        return

    print("\n==============================")
    print("GUROBI SMALL-SIZED BENCHMARK")
    print("==============================")
    print(f"Objective value:          {result['objective']:.6f}")
    print(f"Runtime:                  {result['runtime_seconds']:.4f} seconds")
    print(f"MIP gap:                  {result['mip_gap']:.6f}")
    print(f"Raw waiting component:    {result['raw_waiting_component']:.2f}")
    print(f"Raw overcrowding penalty: {result['raw_overcrowding_component']:.2f}")
    print(f"Raw buses required:       {result['raw_bus_component']:.0f}")

    print("\nSelected frequency-based dispatch plan:")
    print("=" * 150)

    header = (
        f"{'Service':<8} "
        f"{'Period':<15} "
        f"{'Headway':<10} "
        f"{'Freq':<10} "
        f"{'Buses':<8} "
        f"{'Demand':<10} "
        f"{'Demand/Bus':<13} "
        f"{'Cap/Bus':<10} "
        f"{'Total Cap':<12} "
        f"{'Overcrowd':<12} "
        f"{'H Penalty':<10}"
    )

    print(header)
    print("-" * 150)

    for row in result["solution"]:
        print(
            f"{row['service']:<8} "
            f"{row['period']:<15} "
            f"{row['headway_min']:<10.2f} "
            f"{row['frequency_departures']:<10.2f} "
            f"{row['buses_required']:<8} "
            f"{row['demand']:<10.0f} "
            f"{row['demand_per_dispatch']:<13.2f} "
            f"{row['capacity_per_dispatch']:<10.0f} "
            f"{row['total_capacity']:<12.2f} "
            f"{row['overcrowding']:<12.2f} "
            f"{row['headway_penalty']:<10.2f}"
        )

    print("=" * 150)


# ============================================================
# 4. Simple greedy benchmark
# ============================================================

def greedy_frequency_benchmark(
    services,
    periods,
    L,
    available_fleet,
    D,
    q,
    cycle_time,
    headway_options,
    alpha=0.40,
    beta=0.50,
    gamma=0.10,
):
    """
    Simple benchmark heuristic:
    For each service-period, choose the locally best option,
    then repair if fleet availability is exceeded.

    This is not guaranteed optimal.
    It is useful as a baseline to compare against Gurobi.
    """

    def option_cost(i, t, h):
        freq = L[t] / h
        total_cap = freq * q[i]
        over = max(0, D[i, t] - total_cap)
        over_pen = over ** 2
        h_pen = (h - 5) ** 2 + 2
        buses = math.ceil(cycle_time[i] / h)

        return alpha * D[i, t] * h_pen + beta * over_pen + gamma * buses

    selected = {}

    # Step 1: choose locally cheapest option
    for i in services:
        for t in periods:
            best_h = min(
                headway_options[i, t],
                key=lambda h: option_cost(i, t, h)
            )
            selected[i, t] = best_h

    # Step 2: repair fleet violations by increasing headway
    for t in periods:
        while True:
            used_buses = sum(
                math.ceil(cycle_time[i] / selected[i, t])
                for i in services
            )

            if used_buses <= available_fleet[t]:
                break

            best_service_to_relax = None
            lowest_cost_increase = float("inf")

            for i in services:
                current_h = selected[i, t]
                sorted_options = sorted(headway_options[i, t])

                # Try moving to the next larger headway
                larger_options = [h for h in sorted_options if h > current_h]

                if not larger_options:
                    continue

                next_h = larger_options[0]

                old_cost = option_cost(i, t, current_h)
                new_cost = option_cost(i, t, next_h)
                cost_increase = new_cost - old_cost

                if cost_increase < lowest_cost_increase:
                    lowest_cost_increase = cost_increase
                    best_service_to_relax = i

            if best_service_to_relax is None:
                raise RuntimeError(
                    f"Cannot repair fleet violation for period {t}. "
                    f"Increase available fleet or allow larger headways."
                )

            current_h = selected[best_service_to_relax, t]
            larger_options = [
                h for h in sorted(headway_options[best_service_to_relax, t])
                if h > current_h
            ]

            selected[best_service_to_relax, t] = larger_options[0]

    total_cost = 0
    total_buses = 0

    for i in services:
        for t in periods:
            h = selected[i, t]
            total_cost += option_cost(i, t, h)
            total_buses += math.ceil(cycle_time[i] / h)

    return selected, total_cost, total_buses


# ============================================================
# 5. Complexity demonstration
# ============================================================

def complexity_demo(services, periods, headway_options):
    """
    Shows how the number of combinations increases.

    If each service-period has several frequency/headway options,
    total combinations = product of number of options across all service-periods.
    """

    total_combinations = 1

    for i in services:
        for t in periods:
            total_combinations *= len(headway_options[i, t])

    print("\n==============================")
    print("COMPUTATIONAL COMPLEXITY")
    print("==============================")
    print(f"Number of services:       {len(services)}")
    print(f"Number of time periods:   {len(periods)}")
    print(f"Total service-periods:    {len(services) * len(periods)}")
    print(f"Total combinations:       {total_combinations:,}")

    print(
        "\nThis grows exponentially because the model must choose one "
        "frequency/headway option for every service-period pair."
    )


# ============================================================
# 6. Main execution
# ============================================================

if __name__ == "__main__":

    instance = create_small_instance()

    services, periods, L, available_fleet, D, q, cycle_time, headway_options = instance

    result = solve_frequency_based_gurobi(
        services,
        periods,
        L,
        available_fleet,
        D,
        q,
        cycle_time,
        headway_options,
        alpha=0.40,
        beta=0.50,
        gamma=0.10,
        time_limit=60,
        verbose=True,
    )

    print_solution(result)

    selected, greedy_cost, greedy_buses = greedy_frequency_benchmark(
        services,
        periods,
        L,
        available_fleet,
        D,
        q,
        cycle_time,
        headway_options,
        alpha=0.40,
        beta=0.50,
        gamma=0.10,
    )

    print("\n==============================")
    print("GREEDY BENCHMARK")
    print("==============================")
    print(f"Greedy cost:        {greedy_cost:.2f}")
    print(f"Greedy buses used:  {greedy_buses}")
    print("\nGreedy selected headways:")

    for key, value in selected.items():
        print(f"{key}: {value} minutes")

    complexity_demo(services, periods, headway_options)

Set parameter TimeLimit to value 60
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60

Optimize a model with 18 rows, 60 columns and 120 nonzeros (Min)
Model fingerprint: 0xf8acf152
Model has 60 linear objective coefficients
Variable types: 0 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [6e-03, 5e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]

Found heuristic solution: objective 0.2952614
Presolve removed 18 rows and 60 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 0.182391 0.295261 

Optimal solution fou